<a href="https://colab.research.google.com/github/ecuirty-kr/1_DataAnalysis/blob/main/Fin_type_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bigdata_analyst_cert_v2/blob/main/part3/ch6/ch6_ex_type3.ipynb)

### Section1.

In [42]:
#### 단일 표본 검정
#### 문제 조건 : 커피 제조회사 에서 새로 출시한 커피의 카페인 함량이 평균 95 mg 미만이라고 주장.
#### 25 개의 커피 샘플을 무작위 추출. 주장이 타당한지 유의수준 5% 에서 검정
#### 귀무 : u >= 95 mg / 대립 : u < 95 mg

## 1. 표본 데이터 평균
## 2. shapifiro-Wilk  검정의 p-value
## 3. 단일 표본 t-검정의 검정 통계량
## 4. 단일 표본 t-검정의 p-value
## 5. 유의수준 0.05 하에서 귀무가성르 기준으로 검정 결과 채택 / 기각

import pandas as pd
df = pd.DataFrame({
    'Caffeine(mg)': [
        94.2, 93.7, 95.5, 93.9, 94.0, 95.2, 94.7, 93.5, 92.8, 94.4,
        93.8, 94.6, 93.3, 95.1, 94.3, 94.9, 93.9, 94.8, 95.0, 94.2,
        93.7, 94.4, 95.1, 94.0, 93.6
    ]
})
## 단일 표본 검정 (chisquare)
from scipy.stats import ttest_1samp ## 카이제곱 아님 !!
from scipy import stats
print(df['Caffeine(mg)'].mean()) # 결과 : 94.264
print(stats.shapiro(df['Caffeine(mg)'])) # 결과 : ShapiroResult(statistic=np.float64(0.9826578166170536), pvalue=np.float64(0.9322031137746971))
print(ttest_1samp(df['Caffeine(mg)'], 95, alternative='less')) ## TtestResult(statistic=np.float64(-5.501737036221897), pvalue=np.float64(5.8686553916715e-06), df=np.int64(24))
ttest, p_value = ttest_1samp(df['Caffeine(mg)'], 95, alternative='less')
print(ttest, p_value)
print("{:.10f}".format(p_value)) # 0.0000058687  ## 코드 입력할 때 오타 잘 좀 확인하자. 에러 왜 나지 하고 잡고 있음 안 됨..

## 대응 표본 (ttest_rel) 독립 표본 (ttest_ind) - ex. 두 충전기의 성능 비교 (시간 데이터 : 성능이 좋다 = 충전 시간이 덜 걸린다 =값이 작음)

#### 분산 분석 (일원 / 이원) 일원 분산 분석 - 4개 이상의 데이터 (ex. 식물 비료 비교)
### 문제 : 4 개의 중학교에서 다른 교육 방법을 사용해 학생들에게 수학을 가르쳤다. 4개의 교육 방법에 따른 성적 차이가 통계적으로 유의미한지 검정
### 귀무 : 모든 중학교 학생들의 수학 성적 평균은 동일 / 대립 : 모든 중학교 학생들의 수학 성적 평균은 다르다

## 1. 각 그룹 수학 성적에 shapiro-Wilk 검정하여 정규성 확인 , p-value?
## 2. 4개의 수학 성적이 등분산성을 갖는지 확인 Levene 검정 , p-value ?
## 3. 유의수준 0.05 하에서 귀무가설을 기준으로 검정의 결과를 채택 / 기각 선택
## 4. 자유도?
## 5. 잔차의 자유도
## 6. 성적의 제곱 합
## 7. 성적의 평균 제곱
## 8. F-통계량
## 9. 성적에 대한 p-value
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part3/ch6/math.csv")
print(df.head(4)) # 데이터 확인

from scipy import stats  ## shapiro-Wilk 정규성 검정 : (조건 생성 - 조건+대상 shapiro())
cond = df['groups'] == 'group_A'
print(stats.shapiro(df[cond]['scores']))  ## ShapiroResult(statistic=np.float64(0.9715896670696531), pvalue=np.float64(0.9051800443853569)) p-value 확인
cond2 = df['groups'] == 'group_B'
print(stats.levene(df[cond]['scores'], df[cond2]['scores'])) ## 정규성 검정한 모든 값을 levene() 인자로 넣어줘야함(여기서는 생략)
# 결과 : LeveneResult(statistic=np.float64(1.8935064935064936), pvalue=np.float64(0.18568733573055562)) , p-value 확인

# 일원 분산 분석 (ols 모델 생성 - ANOVA Table 생성)
from statsmodels.formula.api import ols
model = ols('scores~groups', data=df).fit()

from statsmodels.stats.anova import anova_lm
print(anova_lm(model)) ## anova_lm(model) * anova_lm()의 인자값으로 ols() model 들어감
# 결과로 테이블 생성
# ANOVA 테이블에서 p-value 확인 : 1.240642e-10 (<0.05 이므로 귀무가설 기각)
# 자유도(df) : 3.0
# 잔차의 자유도(residual : df) :  36.0
# 성적의 제곱합(sum_sq) : 411.8
# 성적의 평균 제곱(mean_sq) : 137.266667
# F-통계량(F) : 34.174274
# p-value : 1.240642e-10 (지수함수 형태를 {:.10f}.format() 하여 실수로 확인 가능)

# 이원 분산 분석 : 요인이 2 가지 이상 - 귀무/대립 가설이 2개 이상임
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part3/ch6/tomato2.csv")

import statsmodels.api as sm #### 얘는 import ~ as 임 (이원분산분석 : statsmodels.api as sm -> sm.stats.anova_lm(model))
from statsmodels.formula.api import ols ## 여기도 ols() 모델 사용
# 분산 분석 수행
model = ols('수확량~C(비료유형)*C(물주기)', data=df).fit()
anova_table = sm.stats.anova_lm(model)
print(anova_table)


### 카이제곱 검정 (적합도 / 독립성 / 동질성)
from scipy.stats import chisquare ## 문제에서 빈도+비율 => 적합도 / 교차표 및 로우데이터 = 독립성 OR 동질성
#### 문제 : 한 도시 운전자 1,000 명 대상으로 교통사고 경험 수 조사
## 적합도 검정은 변수 observed(관측값) , expected(기대값) 두 가지로, 이게 chisquare() 함수 인자로 쓰임
observed = [550, 250, 100, 70, 30]
expected = [1000*0.6, 1000*0.25, 1000*0.08, 1000*0.05, 1000*0.02] ## expected 값이 비율이면 (총 조사인원 * 비율)으로 빈도 수로 변환해야 함
print(chisquare(observed, expected)) ## 귀무 : 전국적 경향을 따름 / 대립 : 전국적 분포를 따르지 않음
# 결과 : Power_divergenceResult(statistic=np.float64(22.166666666666668), pvalue=np.float64(0.00018567620386641427))


## 카이제곱 / 독립성 검정
### 기사시험 캠프에 대한 캠프 소속 여부와 등록 여부 관계 (독립성 검정 : 두 요인이 서로 독립적/연관)
### 귀무 : 캠프와 세미나 등록 여부는 서로 독립적 / 대립 : 캠프와 세미나 등록 여부는 서로 연관(독립적이지 않음)
## 1. 검정 통계량
## 2. p-value
## 3. 채택 / 기각
from scipy.stats import chi2_contingency
observed = pd.DataFrame([[50, 30], [60, 40]]) # 교차표 데이터 주어짐 = 교차표 데이터 보고 새 데이터 프레임 생성
print(chi2_contingency(observed))   ### 데이터 프레임 생성 시 괄호 규칙 준수 : pd.DataFrame([[50, 30], [60, 40]])
# 결과 : statistic=np.float64(0.03535714285714309), pvalue=np.float64(0.8508492527705047), dof=1, expected_freq=array([[48.88888889, 31.11111111], [61.11111111, 38.88888889]])


df = pd.DataFrame({
        '캠프': ['빅분기']*80 + ['정처기']*100,
        '등록여부': ['등록']*50 + ['등록안함']*30 + ['등록']*60 + ['등록안함']*40
}) # 로우 데이터 제공 시 교차표로 변경(pd.crosstab(df['이름'], df['이름']))
df = pd.crosstab(df['캠프'], df['등록여부'])
print(chi2_contingency(df)) # statistic=np.float64(0.03535714285714309), pvalue=np.float64(0.8508492527705047), dof=1, expected_freq=array([[48.88888889, 31.11111111], [61.11111111, 38.88888889]])

###### 다중 선형 회귀 (종속변수 / 독립변수)
## 문제 1. 상관계수 (model.corr())
## 문제 2. 결정계수 (model.rsquared)
## 문제 3. 회귀 계수 (model.params[])
## 문제 4. 절편(Intercept)  (model.summary())
## 문제 5. '온도'의 회귀 계수가 통계적으로 유의미한지 검정
# ******** model.pvalues['온도' ]
## 문제 6. 새 데이터 제공 (할인율 10%, 온도:25)
new_data = pd.DataFrame({
    '할인율' : [10],
    '온도' : [25]
})
model.predict(new_data) ### 아 2유형도 해야 하는데 시간 촉박하다......
## 문제 7. 잔차제곱합
df['잔차'] = df['주문량'] - model.predict(df) ## 실제 - 예측
print(sum(df['잔차']**2))
## 문제 8. MSE
print((df['잔차']**2).mean())
## 문제 9. 온도의 회귀 계수에 대한 90% 신뢰구간
# model.params['온도'].conf_int(alpha=0.1) ### 그냥 모든 변수의 신뢰구간 구하고 '온도' 회귀 계수를 확인하는게 좋음
model.conf_int(alpha=0.1)
## 문제 10. 새 데이터 주어지고 90% 신뢰구간 및 예측구간
new_data = pd.DataFrame({})  # 새 데이터 생성
pred = model.get_predict(new_data)
result = pred.summary_frame(alpha=0.1) ## summary_frame() 인자값은 신뢰구간 (%) alpha 값임!!!! 잊지말기!!!!!!!
print(result)
## 문제 11. 할인율, 온도 고정, 광고비가 배달 주문량에 영향을 주는지 가설 검정. 기각 / 채택
model.pvalues['광고비'] ## < 0.05 인지 확인

############## 로지스틱 회귀 (종속변수 / 독립변수)
# 문제 1. 로지스틱 모델 적합, 유의하지 않은 독립 변수 개수 (model.summary() p-value 값 확인)
from statsmodels.formula.api import logit
model = logit('종속~독립', data=df).fit()
print(model.summary())
## 문제 2. p-value가 0.05 보다 작은 유의한 변수만 사용하여 수정된 모델 만들고 적합, 가장 큰 p-value 변수?
model = logit('종속~독립+독립').fit()
model.pvalues[1:].idxmax() ## Intercept 제외 pvalue 값 가장 큰 데이터
## 문제 3. 절댓값이 가장 큰 회귀계수를 가진 변수 이름
model.params[1:].abs().idxmax()

## 문제 4. 로그 우도 (model.llf)
print(model.llf) # 로그 우도 : .llf
## 문제 5. 잔차 이탈도 (잔차이탈도 : 로그 우도 * -2)
print(model.llf * -2)
## 문제 6. booked 변수가 3 증가할 때 오즈비
import numpy as np
np.exp(model.params['booked']*3)

## 문제 7. p-value 가 0.05 보다 작은 회귀계수의 총합 (상수항도 해당하면 포함)
model.params[model.pvalues<0.05].sum() ## params[] 안에 조건 넣어도 성립
## 문제 8. b 데이터를 사용해 예측, b 데이터 target과 비교한 정확도 계산
from sklearn.metrics import accuracy_score
# 예측 먼저
pred = model.predict(b)
accuracy = model.accuracy_score(b['targer'], pred) ## metrics 함수 인자(df['값'], pred) -> 예측 선수행
print(accuracy)
## 문제 9. 오류율
print((1-accuracy)) ## 오류율 : (1-정확도)

### 데이터 분할 : 요구사항 = 데이터의 앞 50% 와 뒤 50%를 분할하여 a, b로 나누어 작업
middle = len(df/2)
a = df.iloc[:middle]
b = df.iloc[:middle]
print(a.shape, b.shape) # 분할 데이터 크기 확인


94.264
ShapiroResult(statistic=np.float64(0.9826578166170536), pvalue=np.float64(0.9322031137746971))
TtestResult(statistic=np.float64(-5.501737036221897), pvalue=np.float64(5.8686553916715e-06), df=np.int64(24))
-5.501737036221897 5.8686553916715e-06
0.0000058687
    groups  scores
0  group_A      85
1  group_A      88
2  group_A      90
3  group_A      82
ShapiroResult(statistic=np.float64(0.9715896670696531), pvalue=np.float64(0.9051800443853569))
LeveneResult(statistic=np.float64(1.8935064935064936), pvalue=np.float64(0.18568733573055562))
            df  sum_sq     mean_sq          F        PR(>F)
groups     3.0   411.8  137.266667  34.174274  1.240642e-10
Residual  36.0   144.6    4.016667        NaN           NaN
                  df        sum_sq      mean_sq         F    PR(>F)
C(비료유형)          2.0   5251.722222  2625.861111  3.184685  0.059334
C(물주기)           3.0   9057.000000  3019.000000  3.661490  0.026460
C(비료유형):C(물주기)   6.0   4271.833333   711.972222  0.863491  0.53542

PatsyError: predict requires that you use a DataFrame when predicting from a model
that was created using the formula api.

The original error message returned by patsy is:
Error evaluating factor: NameError: name '비료유형' is not defined
    수확량~C(비료유형)*C(물주기)
        ^^^^^^^

### Section9.

In [49]:
######################### 작업형 2 **** concat 다시 해보기
## 문제 조건 : 이직 여부 예측
# 예측 컬럼 : target (0: 새 일자리를 찾지 않음 / 1: 새 일자리를 찾음)
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch4/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch4/test.csv") ### 예전 아무 문제에서나 가져온 train, test

## 아 유형 1 어떡하지
# 데이터 확인
print(train.head(4), test.head(4))

# 2. 탐색적 데이터 분석 (크기 / 자료형 / 결측치 / target 빈도 수)
print(train.shape, test.shape) # (6665, 11) (2154, 10)
print(train.info(), test.info()) # float64(2), int64(2), object(6) / target = int (train)
print(train.isnull().sum(), test.isnull().sum()) # 없음
print(train['Gender'].value_counts()) # 대충 1:1 - 균형 데이터 (아무거나 가져온거라 의미 X)

# 3. 데이터 전처리 (인코딩 필요/ 결측치 있으면 결측치 처리) - 원-핫 인코딩 / 레이블 인코딩
target = train.pop('Gender')
## 결측치 있으면 df['변수'] = df['변수'].fillna('대체값')
# # 1. 원-핫 기본
train = pd.get_dummies(train)
test = pd.get_dummmies(test)
print(train.shape, test.shape) # 열 개수 동일한지 확인

# # 2. 원-핫 결합
combined = pd.concat([train, test])
combined_dummies = pd.get_dummmies(combined) ## get_dummies 수행 후 분할
n_train = len(train)
train = combined_dummies[:n_train]
test = combined_dummies[n_train:]

# # 3. 레이블 인코딩
from sklearn.preprocessing import LabelEncoder
cols = train.select_dtypes(include='O').columns
for col in cols:
  le = LabelEncoder()
  train[col] = le.fit_transform(train[col])
  test[col] = le.transform(test[col])
print(train.shape, test.shape)

## 4. 검증 데이터 분리
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

## 5. 머신러닝 학습 및 평가
## 시간이 없다.
## 5-1. 랜덤포레스트(지금은 분류 / 회귀)
from sklearn.ensemble import RandomForestClasifier

from sklearn.metrics import f1_score ### 가야함. 시험장. 아.
# 평가
## 5-2. LightGBM
import lightgbm as lgb
lg = lgb.LGBMClasifier(random_state=0, verbose=-1)
## 평가
## 평가 비교 성능 계산

## 5. 예측 및 결과 파일 생성
pred = lg.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv("result.csv", index=False)
print(pred.shape)
file = pd.read_csv("result.csv")
print(file)



       ID  Gender Ever_Married  Age Graduated  Profession  Work_Experience  \
0  462809    Male           No   22        No  Healthcare              1.0   
1  466315  Female          Yes   67       Yes    Engineer              1.0   
2  461735    Male          Yes   67       Yes      Lawyer              0.0   
3  461319    Male          Yes   56        No      Artist              0.0   

  Spending_Score  Family_Size  Var_1  Segmentation  
0            Low          4.0  Cat_4             4  
1            Low          1.0  Cat_6             2  
2           High          2.0  Cat_6             2  
3        Average          2.0  Cat_6             3          ID  Gender Ever_Married  Age Graduated  Profession  Work_Experience  \
0  458989  Female          Yes   36       Yes    Engineer              0.0   
1  458994    Male          Yes   37       Yes  Healthcare              8.0   
2  459000    Male          Yes   59        No   Executive             11.0   
3  459003    Male          Yes  

AttributeError: module 'pandas' has no attribute 'get_dummmies'